# Train v5a — Drops Only
**Changes vs v4:**
- Drop 11 features ranked in bottom 30% SHAP for ALL horizons
- Drop per-horizon features with high gain importance but low SHAP
- No monotone constraints


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))


In [ ]:
import lightgbm as lgb
import pandas as pd
import numpy as np
import gc
from src.score import weighted_rmse_score as score


In [3]:
def set_cat(df, cat_features):
    for feat in cat_features:
        df[feat] = df[feat].astype('category')
    return df


## Config

In [4]:
HORIZONS = [1, 3, 10, 25]

ZERO_CODES = ['MLAAMU3K', 'EP12UF2K', '1HEMHZK2', 'SJZP0OVU', '83EG83KQ']

SPLIT_INDEX = 3500

VERSION = 'v5a'

params = {
    'objective': 'regression',
    'boosting_type': 'gbdt',
    'metric': 'rmse',
    'num_leaves': 80,
    'min_child_samples': 200,
    'lambda_l2': 10,
    'learning_rate': 0.01,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'feature_fraction': 0.8,
    'seed': 42,
    'verbosity': -1
}

# ── FEATURE DROPS ──
# Global: bottom 30% SHAP for ALL horizons
GLOBAL_DROP = [
    'feature_aa', 'feature_au', 'feature_ax', 'feature_ba', 'feature_bb',
    'feature_bc', 'feature_bj', 'feature_bv', 'feature_c', 'feature_e',
    'feature_f'
]

# Per-horizon: high gain importance but low SHAP mean
PER_HORIZON_DROP = {
    1:  ['feature_ce', 'feature_cf', 'feature_x', 'feature_y'],
    3:  ['feature_ce', 'feature_cf', 'feature_y', 'feature_z'],
    10: ['feature_ce', 'feature_cf'],
    25: ['feature_cf'],
}


## Load & Prepare Data

In [ ]:
df = pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
cat_features = ['code', 'sub_category', 'horizon']

# Base features (same as v4)
all_features = [feat for feat in df.columns
                if feat not in ['id', 'horizon','sub_code', 'ts_index', 'weight', 'y_target']]

# Remove global drops from base feature list
base_features = [f for f in all_features if f not in GLOBAL_DROP]
print(f"Features: {len(all_features)} -> {len(base_features)} after global drops")
print(f"Dropped globally: {GLOBAL_DROP}")

df = set_cat(df, cat_features)


Features: 88 -> 77 after global drops
Dropped globally: ['feature_aa', 'feature_au', 'feature_ax', 'feature_ba', 'feature_bb', 'feature_bc', 'feature_bj', 'feature_bv', 'feature_c', 'feature_e', 'feature_f']


## Time-based Train/Val Split

In [7]:
train_df = df[df['ts_index'] <= SPLIT_INDEX]
val_df   = df[df['ts_index'] >  SPLIT_INDEX]
print(f"Train size: {len(train_df):,} | Val size: {len(val_df):,} | Split ts_index: {SPLIT_INDEX}")
del df
gc.collect()


Train size: 5,172,152 | Val size: 165,262 | Split ts_index: 3500


20

## Train One Model per Horizon (with per-horizon feature drops)

In [ ]:
models      = {}
importances = {}
features_used = {}  # track features per horizon

for h in HORIZONS:
    print(f"\n{'='*60}")
    print(f"  Training horizon = {h}")
    print(f"{'='*60}")

    # Per-horizon feature list: base - per_horizon_drops
    h_drops = PER_HORIZON_DROP.get(h, [])
    features = [f for f in base_features if f not in h_drops]
    features_used[h] = features
    print(f"  Features: {len(base_features)} -> {len(features)} after per-horizon drops")
    print(f"  Dropped for H={h}: {h_drops}")

    # Update cat_features for this horizon (only keep those still in features)
    h_cat = [c for c in cat_features if c in features]

    # Subset by horizon
    h_train = train_df[train_df['horizon'] == h].copy()
    h_val   = val_df[val_df['horizon'] == h].copy()
    print(f"  Train rows: {len(h_train):,} | Val rows: {len(h_val):,}")

    # LightGBM Datasets
    dtrain = lgb.Dataset(
        h_train[features],
        label=h_train['y_target'],
        weight=h_train['weight'],
        categorical_feature=h_cat,
        free_raw_data=False
    )
    dval = lgb.Dataset(
        h_val[features],
        label=h_val['y_target'],
        weight=h_val['weight'],
        categorical_feature=h_cat,
        reference=dtrain,
        free_raw_data=False
    )

    # Train (same params as v4, no monotone constraints)
    model = lgb.train(
        params,
        dtrain,
        valid_sets=[dtrain, dval],
        valid_names=['train', 'val'],
        num_boost_round=4200,
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=100)
        ]
    )
    models[h] = model

    # Save model
    model_path = PROJECT_ROOT / "models" / f"lgb_model_{VERSION}_horizon{h}.txt"
    model.save_model(model_path, num_iteration=model.best_iteration)
    print(f"  Model saved -> {model_path}  (best iteration: {model.best_iteration})")

    # Feature importance
    imp = pd.DataFrame({
        'feature': features,
        'importance': model.feature_importance(importance_type='gain')
    }).sort_values('importance', ascending=False)
    importances[h] = imp
    print(f"\n  Top-10 features (horizon={h}):")
    print(imp.head(10).to_string(index=False))



  Training horizon = 1
  Features: 77 -> 73 after per-horizon drops
  Dropped for H=1: ['feature_ce', 'feature_cf', 'feature_x', 'feature_y']
  Train rows: 1,351,193 | Val rows: 43,460
Training until validation scores don't improve for 200 rounds
[100]	train's rmse: 0.000976547	val's rmse: 0.0011356
[200]	train's rmse: 0.00097111	val's rmse: 0.00113504
[300]	train's rmse: 0.00096704	val's rmse: 0.00113476
[400]	train's rmse: 0.000963767	val's rmse: 0.00113509
[500]	train's rmse: 0.000960714	val's rmse: 0.00113492
Early stopping, best iteration is:
[311]	train's rmse: 0.000966694	val's rmse: 0.00113463
  Model saved -> /home/lingod/kaggle_projects/ts_forecasting/models/lgb_model_v5a_horizon1.txt  (best iteration: 311)

  Top-10 features (horizon=1):
     feature   importance
  feature_al 2.121983e+06
  feature_by 1.466061e+06
   feature_s 1.352656e+06
sub_category 1.182133e+06
  feature_cb 8.863249e+05
   feature_z 8.540669e+05
   feature_t 8.359441e+05
  feature_cg 8.269792e+05
  feat

## Train Scoring (per-horizon + overall)

In [9]:
train_preds_series = pd.Series(index=train_df.index, dtype=float)

for h in HORIZONS:
    h_train = train_df[train_df['horizon'] == h].copy()
    model   = models[h]
    features = features_used[h]

    preds = model.predict(h_train[features], num_iteration=model.best_iteration)
    preds = pd.Series(preds, index=h_train.index)

    zero_mask = h_train['code'].isin(ZERO_CODES)
    preds[zero_mask] = 0.0
    train_preds_series[h_train.index] = preds

    h_score = score(h_train['y_target'], preds, h_train['weight'])
    print(f"  Horizon {h:>2d} | Train Score: {h_score:.4f}  (zero-coded rows: {zero_mask.sum():,})")

overall_train_score = score(train_df['y_target'], train_preds_series, train_df['weight'])
print(f"\n  Overall Train Score: {overall_train_score:.4f}")


  Horizon  1 | Train Score: 0.1740  (zero-coded rows: 266,358)
  Horizon  3 | Train Score: 0.2386  (zero-coded rows: 265,104)
  Horizon 10 | Train Score: 0.2875  (zero-coded rows: 255,715)
  Horizon 25 | Train Score: 0.4868  (zero-coded rows: 230,583)

  Overall Train Score: 0.3982


## Validation Scoring (per-horizon + overall)

In [10]:
val_preds_series = pd.Series(index=val_df.index, dtype=float)

for h in HORIZONS:
    h_val  = val_df[val_df['horizon'] == h].copy()
    model  = models[h]
    features = features_used[h]

    preds = model.predict(h_val[features], num_iteration=model.best_iteration)
    preds = pd.Series(preds, index=h_val.index)

    zero_mask = h_val['code'].isin(ZERO_CODES)
    preds[zero_mask] = 0.0
    val_preds_series[h_val.index] = preds

    h_score = score(h_val['y_target'], preds, h_val['weight'])
    print(f"  Horizon {h:>2d} | Val Score: {h_score:.4f}  (zero-coded rows: {zero_mask.sum():,})")

overall_val_score = score(val_df['y_target'], val_preds_series, val_df['weight'])
print(f"\n  Overall Val Score: {overall_val_score:.4f}")


  Horizon  1 | Val Score: 0.0634  (zero-coded rows: 7,675)
  Horizon  3 | Val Score: 0.0952  (zero-coded rows: 7,618)
  Horizon 10 | Val Score: 0.1756  (zero-coded rows: 7,238)
  Horizon 25 | Val Score: 0.2190  (zero-coded rows: 6,609)

  Overall Val Score: 0.1871


In [11]:
del train_df, val_df, train_preds_series, val_preds_series
gc.collect()


68

## Test Predictions

In [ ]:
df_test = pd.read_parquet(PROJECT_ROOT / "data" / "test.parquet")
df_test = set_cat(df_test, cat_features)


In [13]:
test_preds_series = pd.Series(index=df_test.index, dtype=float)

for h in HORIZONS:
    h_test    = df_test[df_test['horizon'] == h]
    model     = models[h]
    features  = features_used[h]

    preds     = model.predict(h_test[features], num_iteration=model.best_iteration)
    preds     = pd.Series(preds, index=h_test.index)

    zero_mask = h_test['code'].isin(ZERO_CODES)
    preds[zero_mask] = 0.0

    test_preds_series[h_test.index] = preds
    print(f"  Horizon {h:>2d} | Test rows: {len(h_test):,}  (zero-coded rows: {zero_mask.sum():,})")

assert test_preds_series.isna().sum() == 0, "Some test rows were not predicted!"
print(f"\n  Total test predictions: {len(test_preds_series):,}")


  Horizon  1 | Test rows: 379,617  (zero-coded rows: 69,005)
  Horizon  3 | Test rows: 376,558  (zero-coded rows: 68,523)
  Horizon 10 | Test rows: 362,057  (zero-coded rows: 65,675)
  Horizon 25 | Test rows: 328,875  (zero-coded rows: 59,033)

  Total test predictions: 1,447,107


## Build & Save Submission

In [14]:
prediction_df = pd.DataFrame({
    'id':         df_test['id'].values,
    'prediction': test_preds_series.values
})

prediction_df.info(max_cols=10)
prediction_df.head()


<class 'pandas.DataFrame'>
RangeIndex: 1447107 entries, 0 to 1447106
Data columns (total 2 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   id          1447107 non-null  str    
 1   prediction  1447107 non-null  float64
dtypes: float64(1), str(1)
memory usage: 73.8 MB


,id,prediction
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647,-0.031091
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647,-0.066085
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647,-0.556001
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.003741
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648,-0.075656


In [ ]:
prediction_df.to_csv(
    PROJECT_ROOT / "submissions" / f"submission_{VERSION}.csv",
    index=False
)
print(f"Submission saved: submission_{VERSION}.csv")


Submission saved: submission_v5a.csv


## Comparison vs v4
| Metric | v4 | v5a (drops only) |
|--------|----|------------------|
| Val H1 | 0.0642 | 0.0634 |
| Val H3 | 0.0966 | 0.0952 |
| Val H10 | 0.2014 | 0.1756 |
| Val H25 | 0.2217 |0.2190 |
| **Overall Val** | **0.1962** |0.1871 |